# Phase 9: GRPO Fraud Decision Policy

**Phase:** 18

Implement a controlled group-relative policy optimization experiment.

This notebook preserves the project contract established in Phase 1 and Phase 2:

- Permanent code in GitHub
- Permanent dataset and artifacts in Google Drive
- ULB `creditcard.csv`
- Chronological 60/20/20 partitions
- Training-only preprocessing
- Validation-only model and threshold selection
- One final test evaluation
- Reproducible artifacts and acceptance checks

> Research POC only. Do not use the outputs for live transaction decisions.


## Storage and execution model

- GitHub repository: `deepak-malviyaa/enterprise-adaptive-learning-lab`
- Dataset: `MyDrive/enterprise-adaptive-learning-lab-data/raw/creditcard.csv`
- Runs: `MyDrive/enterprise-adaptive-learning-lab-data/outputs/runs/`
- Colab `/content`: temporary runtime only


In [ ]:
# Common setup: Drive, GitHub, project installation, and paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, random, shutil, subprocess, hashlib, platform

SEED=42
random.seed(SEED)
REPO_URL='https://github.com/deepak-malviyaa/enterprise-adaptive-learning-lab.git'
PROJECT_ROOT=Path('/content/enterprise-adaptive-learning-lab')
DRIVE_ROOT=Path('/content/drive/MyDrive/enterprise-adaptive-learning-lab-data')
DATA_PATH=DRIVE_ROOT/'raw'/'creditcard.csv'
RUNS_ROOT=DRIVE_ROOT/'outputs'/'runs'
RUNS_ROOT.mkdir(parents=True,exist_ok=True)

if (PROJECT_ROOT/'.git').exists():
    result=subprocess.run(['git','-C',str(PROJECT_ROOT),'pull','--ff-only'],text=True,capture_output=True)
elif PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
    result=subprocess.run(['git','clone',REPO_URL,str(PROJECT_ROOT)],text=True,capture_output=True)
else:
    result=subprocess.run(['git','clone',REPO_URL,str(PROJECT_ROOT)],text=True,capture_output=True)
print(result.stdout); print(result.stderr)
if result.returncode!=0: raise RuntimeError('Git operation failed.')

pyproject=PROJECT_ROOT/'pyproject.toml'
text=pyproject.read_text(encoding='utf-8').replace('>=3.10,<3.13','>=3.10,<3.14')
pyproject.write_text(text,encoding='utf-8')
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
if not DATA_PATH.exists(): raise FileNotFoundError(f'Missing permanent dataset: {DATA_PATH}')
print('Project and dataset ready.')


In [ ]:
# Common data contract, checksum, chronological split, and preprocessing
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score,roc_auc_score,precision_score,recall_score,f1_score,brier_score_loss,confusion_matrix

np.random.seed(SEED)
def sha256_file(path):
    d=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): d.update(b)
    return d.hexdigest()

DATASET_SHA256=sha256_file(DATA_PATH)
df=pd.read_csv(DATA_PATH)
EXPECTED=['Time']+[f'V{i}' for i in range(1,29)]+['Amount','Class']
missing=sorted(set(EXPECTED)-set(df.columns))
if missing or set(df.Class.unique())!={0,1}: raise ValueError(f'Invalid dataset. Missing={missing}')

df=df.sort_values('Time',kind='mergesort').reset_index(drop=True)
df['row_id']=np.arange(len(df))
a=int(len(df)*.60); b=int(len(df)*.80)
train_df=df.iloc[:a].copy(); val_df=df.iloc[a:b].copy(); test_df=df.iloc[b:].copy()
FEATURES=['Time']+[f'V{i}' for i in range(1,29)]+['Amount']
X_train=train_df[FEATURES].copy(); y_train=train_df.Class.to_numpy()
X_val=val_df[FEATURES].copy(); y_val=val_df.Class.to_numpy()
X_test=test_df[FEATURES].copy(); y_test=test_df.Class.to_numpy()
scaler=StandardScaler().fit(X_train[['Time','Amount']])
for frame in [X_train,X_val,X_test]: frame.loc[:,['Time','Amount']]=scaler.transform(frame[['Time','Amount']])
X_train=X_train.to_numpy(np.float32); X_val=X_val.to_numpy(np.float32); X_test=X_test.to_numpy(np.float32)
print('Checksum:',DATASET_SHA256); print(X_train.shape,X_val.shape,X_test.shape)

def metrics(y,p,threshold=.5):
    pred=(p>=threshold).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'pr_auc':float(average_precision_score(y,p)),'roc_auc':float(roc_auc_score(y,p)),'precision':float(precision_score(y,pred,zero_division=0)),'recall':float(recall_score(y,pred,zero_division=0)),'f1':float(f1_score(y,pred,zero_division=0)),'brier':float(brier_score_loss(y,p)),'threshold':float(threshold),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}

def select_threshold(y,p):
    rows=[]
    for t in np.arange(.01,1,.01): rows.append(metrics(y,p,float(t)))
    return max(rows,key=lambda r:(r['f1'],r['recall']))


In [ ]:
# PyTorch runtime, loaders, weighted BCE, focal loss, training, and evaluation
subprocess.run([sys.executable,'-m','pip','install','-q','torch'],check=True)
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader

torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE=1024 if DEVICE.type=='cuda' else 512
train_loader=DataLoader(TensorDataset(torch.tensor(X_train),torch.tensor(y_train,dtype=torch.float32)),batch_size=BATCH_SIZE,shuffle=True)
val_tensor=torch.tensor(X_val,dtype=torch.float32,device=DEVICE)
test_tensor=torch.tensor(X_test,dtype=torch.float32,device=DEVICE)
pos_weight=torch.tensor([(y_train==0).sum()/(y_train==1).sum()],device=DEVICE,dtype=torch.float32)

class FocalLoss(nn.Module):
    def __init__(self,alpha=.75,gamma=2.): super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self,logits,target):
        bce=nn.functional.binary_cross_entropy_with_logits(logits,target,reduction='none')
        p=torch.sigmoid(logits); pt=torch.where(target==1,p,1-p); at=torch.where(target==1,self.alpha,1-self.alpha)
        return (at*(1-pt).pow(self.gamma)*bce).mean()

def predict(model,tensor):
    model.eval()
    with torch.no_grad(): return torch.sigmoid(model(tensor)).cpu().numpy()

def train_model(model,loss_fn,epochs=25,patience=5,lr=1e-3):
    model=model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    best=-1.; best_state=None; stale=0; history=[]
    for epoch in range(1,epochs+1):
        model.train(); total=0.
        for xb,yb in train_loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE); opt.zero_grad(set_to_none=True)
            logits=model(xb); loss=loss_fn(logits,yb); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5.); opt.step(); total+=loss.item()*len(xb)
        vp=predict(model,val_tensor); score=average_precision_score(y_val,vp); history.append({'epoch':epoch,'loss':total/len(train_loader.dataset),'val_pr_auc':score})
        if score>best: best=score; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
        else: stale+=1
        if stale>=patience: break
    model.load_state_dict(best_state); return model,pd.DataFrame(history)
print('Device:',DEVICE,'Batch:',BATCH_SIZE)


## GRPO contract

For each transaction, the old policy samples a group of actions. Rewards are normalized within that group. The policy is updated using a clipped probability ratio. Groups with near-zero reward variance provide no meaningful relative signal and are tracked explicitly.


In [ ]:
ACTIONS=4; GROUP_SIZE=8; CLIP=.2; KL_BETA=.02; ENTROPY_BETA=.01
REWARD=torch.tensor([[1,-2,-3,-10],[-25,5,7,12]],dtype=torch.float32,device=DEVICE)
class Policy(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Linear(30,128),nn.Tanh(),nn.Linear(128,64),nn.Tanh(),nn.Linear(64,ACTIONS))
    def forward(self,x): return self.net(x)
policy=Policy().to(DEVICE); reference=Policy().to(DEVICE); reference.load_state_dict(policy.state_dict()); opt=torch.optim.AdamW(policy.parameters(),3e-4); history=[]
for epoch in range(15):
    zero_var=0; batches=0; epoch_reward=[]
    for xb,yb in train_loader:
        xb=xb.to(DEVICE); yb=yb.long().to(DEVICE)
        with torch.no_grad(): old_logits=policy(xb); old_probs=torch.softmax(old_logits,-1); dist=torch.distributions.Categorical(old_probs); actions=dist.sample((GROUP_SIZE,)).T; old_logp=torch.log(old_probs.gather(1,actions)+1e-9); rewards=REWARD[yb.unsqueeze(1),actions]; std=rewards.std(1,keepdim=True); zero_var+=(std<1e-6).sum().item(); advantages=(rewards-rewards.mean(1,keepdim=True))/(std+1e-6)
        logits=policy(xb); probs=torch.softmax(logits,-1); logp=torch.log(probs.gather(1,actions)+1e-9); ratio=torch.exp(logp-old_logp); surrogate=torch.minimum(ratio*advantages,torch.clamp(ratio,1-CLIP,1+CLIP)*advantages).mean(); entropy=-(probs*torch.log(probs+1e-9)).sum(1).mean()
        with torch.no_grad(): ref_probs=torch.softmax(reference(xb),-1)
        kl=(probs*(torch.log(probs+1e-9)-torch.log(ref_probs+1e-9))).sum(1).mean(); loss=-surrogate+KL_BETA*kl-ENTROPY_BETA*entropy
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(policy.parameters(),5.); opt.step(); epoch_reward.append(rewards.mean().item()); batches+=len(xb)
    history.append({'epoch':epoch+1,'mean_group_reward':float(np.mean(epoch_reward)),'zero_variance_fraction':zero_var/batches,'kl':float(kl.item()),'entropy':float(entropy.item())})
policy.eval()
with torch.no_grad(): actions=policy(test_tensor).argmax(-1).cpu().numpy()
rewards=np.array([[1,-2,-3,-10],[-25,5,7,12]])[y_test,actions]
final={'mean_reward':float(rewards.mean()),'fraud_approval_rate':float((actions[y_test==1]==0).mean()),'false_decline_rate':float((actions[y_test==0]==3).mean()),'action_distribution':np.bincount(actions,minlength=4).tolist()}
print(final)
RUN_DIR=RUNS_ROOT/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_phase9_grpo'); RUN_DIR.mkdir(parents=True); torch.save(policy.state_dict(),RUN_DIR/'policy.pt'); pd.DataFrame(history).to_csv(RUN_DIR/'history.csv',index=False); (RUN_DIR/'metrics.json').write_text(json.dumps(final,indent=2))


## Next

The final research notebook combines feature tokenization, Transformer encoding, expert routing, and GRPO. It must be compared with cross-entropy and focal-loss MoE baselines.
